# Boosting

Boosting
: Combines many weak classifiers (that are slightly better than random guessing) to produce one powerful committee. 
- Weak classifiers included in the final model do not hwave equal weights. 

$$G(x) = \operatorname{sign}\!\left(\sum_{m=1}^{M} a_m \, G_m(x)\right),$$
$$G_m(x) = \text{ a weak learner, } a_m = \text{ weights}$$

## Adaboost
- Adaboost is one of the first boosting model with binary targets, $(-1,1)$.
- Fit a weak classifier, $G_m(x)$ on weighted data, then increase weights on misclassified points so that the next round focuses on them. 
### Process
1. Initailize weights, $w_i=\frac{1}{N}, i=1, ...N$
2. For $m_1$ to $M$:
  a. Fit a classifier $G_m(x)$ to the training data using weights $w_i$. 
  b. Compute $$\mathrm{err}_m \;=\; \frac{\sum_{i=1}^{N} w_i \, I\!\big(y_i \neq G_m(x_i)\big)}{\sum_{i=1}^{N} w_i}$$
    - This is the _weighted fraction_ of incorrectly classified observations.
  c. Compute $$\alpha_m \;=\; \log\!\left(\frac{1-\mathrm{err}_m}{\mathrm{err}_m}\right)$$
  d. Set $$w_i \leftarrow w_i \exp\!\left[\alpha_m \, I\!\big(y_i \neq G_m(x_i)\big)\right]$$
3. Final classification: $$G(x) \;=\; \operatorname{sign}\!\left(\sum_{m=1}^{M} \alpha_m \, G_m(x)\right)$$

- If classification is done correctly, the weight of an observation does not change. 
- If classification is done incorrectly, the weight is increased by multiplying with $\exp{(\alpha_m)}$.
  - $\alpha$ varies with the degree of misclassification. 
  - $\alpha$ is _large_ if the error is _small_.

:::{tip} Adaboost Example
:class: dropdown
|  x  |  y  | m=0 w  | m=1 w  |
|-----|-----|--------|--------|
|  0  | -1  | 0.1667 | =w     |
|  3  | +1  | 0.1667 | =w     |
|  5  | -1  | 0.1667 | =w*2   |
|  8  | -1  | 0.1667 | =w*2   |
|  8  | +1  | 0.1667 | =w     |
| 10  | +1  | 0.1667 | =w     |

For $m=1$, assume that we have a tree stump.
$$
G_1(x)=
\begin{cases}
-1 & \text{if } x < 1.5 \\
+1 & \text{if } x \ge 1.5
\end{cases}
$$

$$\mathrm{err}_m \;=\; \frac{\sum_{i=1}^{N} w_i \, I\!\big(y_i \neq G_m(x_i)\big)}{\sum_{i=1}^{N} w_i} \rightarrow \text{err}_1 = \frac{1}{6} \times 2 = \frac{1}{3}$$

| x  | y  | G₁(x) | correct? |
|----|----|--------|----------|
| 0  | -1 | -1     | Yes       |
| 3  | +1 | +1     | Yes      |
| 5  | -1 | +1     | No      |
| 8  | -1 | +1     | No       |
| 8  | +1 | +1     | Yes       |
| 10 | +1 | +1     | Yes      |

$x=5$ and $x=8$ are misclassified. Retrieve weights (from $m=0$) of misclassified points. 

$$\text{err}_1 = \frac{\frac{1}{6} + \frac{1}{6}}{\frac{1}{6} \times 6} = \frac{1}{3}$$
$$\alpha_1 = \log\left(\frac{1 - \text{err}_m}{\text{err}_m}\right) = \log\left(\frac{1 - \frac{1}{3}}{\frac{1}{3}}\right) = \log(2)$$

Now, increase the weights of $x=5$ and $x=8$ by $\exp(\alpha_1) = \exp\left(\log(2)\right) = 2$.
Incorrectly classified items now have higher weights, so the next classifier is forced to pay more attention to them.

:::

## Forward Stagewise Additive Modeling
$$f(x) = \sum^M_{m=1}\beta_mb(x;\gamma_m),$$
$$b(\space) = \text{ base functions of weak learners, } \gamma = \text{ a parameter that tunes the basis function.}$$
- Generally, boosting fits an additive model. Unlike Adaboost, it is not limited to 2-class classification.
- The formula above is a re-written Adaboost classifier.


### Estimation
- A _loss function_ has to be minimized:
$$\min_{\{\beta_m, \gamma_m\}_1^M} \sum_{i=1}^{N} L\left(y_i, \sum_{m=1}^{M} \beta_m b(x; \gamma_m)\right),$$
$$L = \text{ loss function}$$
- For example, if we set squared error loss as the loss function, 
$$\min_{\{\beta_m, \gamma_m\}_1^M} \sum_{i=1}^{N} \left(y_i - \sum_{m=1}^{M} \beta_m b(x; \gamma_m)\right)^2$$
- In each iteration, we find the best fit to the _residual_ from the previous iteration.

Loss Function
: How wrong are we to predict $f_{m-1}(x_i) + \beta_m b(x_i;\gamma_m)$ when the truth is $y_i$? 

#### Process
1. Initialize $f_0(x) = 0$
2. For $m=1$ to $M$ (total number of iterations),
  Add to the existing model such that the loss function is minimized.
   a. Minimize the loss $\sum^{N}_{i=1}L[y_i, f_{m-1}(x_i) + \beta_mb(x_i;\gamma_m)]$.
   b. Update the function $f_m = f_{m-1}(x) + \beta_mb(x;\gamma_m)$.
      - $b(x;\gamma_m)$: The _new_ weak learner fitted this round.
      - $\beta_m$: How much you trust the new learner.
      - $\gamma_m$: The parameters of the new weak learner. 

## Logit Boost
- **Forward stagewise** algoirthm requires an estimate $f_{m-1}(x_i)$.
- Loss function used is the deviance (negative binomial likelihood).
$$\log L = \sum_{i=1}^{N} \left[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \right]$$
- First, derive $p$ as an expression of $f(x)$. 

$$
\begin{aligned}
\log\left(\frac{p}{1-p}\right) &= f(x) \\
\frac{p}{1-p} &= e^{f(x)} \\
p &= (1-p)e^{f(x)} \\
p &= e^{f(x)} - p e^{f(x)} \\
p + p e^{f(x)} &= e^{f(x)} \\
p(1 + e^{f(x)}) &= e^{f(x)} \\
p &= \frac{e^{f(x)}}{1 + e^{f(x)}}
\end{aligned}
$$

- Rewrite the log likelihood:
$$
\begin{aligned}
p &= \frac{e^{f(x)}}{1 + e^{f(x)}} \\
\ell(y) &\propto \sum_{i=1}^{n} \left[y_i \log(p_i) + (1-y_i)\log(1-p_i)\right] \\
&= \sum_{i=1}^{n} \Bigg[
y_i f(x_i) - y_i \log\!\big(1+e^{f(x_i)}\big)
+ (1-y_i)\log\!\left(\frac{1}{1+e^{f(x_i)}}\right)
\Bigg] \\
&= \sum_{i=1}^{n} \Big[
y_i f(x_i) - \log\!\big(1+e^{f(x_i)}\big)
\Big]
\end{aligned}
$$

- $y$ is coded as $0,1$. 